In [ ]:
# Check installed langchain-related package versions and print a fix suggestion
import sys
from importlib import metadata

def version_of(pkg):
    try:
        return metadata.version(pkg)
    except Exception:
        return None

lc = version_of('langchain')
lcc = version_of('langchain-core')
lco = version_of('langchain-openai')

print(f'langchain={lc}, langchain-core={lcc}, langchain-openai={lco}, python={sys.version.split()[0]}')

if lcc and lc:
    # If langchain-core is 1.x while langchain is 0.3.x, recommend aligning versions
    if lcc.startswith('1.') and lc and lc.startswith('0.3'):
        print('\nDetected incompatible combination: langchain (0.3.x) with langchain-core (1.x).')
        print('Recommended fix (run in the dev container shell):')
        print("pip3 install --upgrade 'langchain-core==0.3.78' 'langchain==0.3.27' 'langchain-openai==0.3.35'")
        print('\nOr choose to upgrade langchain packages to matching 1.x releases if available.')
    else:
        print('\nNo obvious mismatch detected between langchain and langchain-core.')

print('\nAfter installing packages, restart the kernel/REPL to pick up the changes.')

In [ ]:
# Updated import path for the chat model
from langchain.chat_models import ChatOpenAI
from os import environ
# Optional: print version to help debug if needed
import langchain
print('langchain version:', getattr(langchain, '__version__', 'unknown'))

In [ ]:
# DO NOT commit your API key into the notebook.
# Put your key into the environment instead (export OPENAI_API_KEY="sk-...")
environ['OPENAI_API_KEY'] = ""  # put your key here (leave blank in committed notebooks)
environ['OPENAI_BASE_URL'] = 'https://api.ai.it.cornell.edu'

llm = ChatOpenAI(
    model="openai.gpt-4o",
    temperature=0.2,
)

<h2>Load Source Text</h2>

In [ ]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("./data/RAG_source.txt")
documents = loader.load()

In [ ]:
documents[0].metadata

In [ ]:
print(documents[0].page_content)

<h2>Split the document</h2>

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
chunk_size = 200
chunk_overlap = 0

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = chunk_size,
    chunk_overlap = chunk_overlap
)

In [ ]:
chunks = text_splitter.split_documents(documents)

In [ ]:
for chunk in chunks:
    print(chunk.page_content)
    print("-----")

## Index chunks into a vector db (ChromaDB)

In [ ]:
# Updated import paths for embeddings and chroma vectorstore
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma


In [ ]:
vectorstore = Chroma.from_documents(documents=chunks, embedding=OpenAIEmbeddings(model="openai.text-embedding-3-large"))

## Test Similarity Search

In [ ]:
vectorstore.similarity_search("what is Zelomax?")

In [ ]:
vectorstore.similarity_search_with_score("what is Zelomax?")

## Setup retrieval

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 20})

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
format_docs(retriever.invoke("what is Zelomax?"))

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
    You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. 
    If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
    
    Question: {question} 
    
    Context: {context} 
    
    Answer:
"""
prompt = PromptTemplate.from_template(template)

## Build RAG chain

In [ ]:
from langchain_core.documents import Document
from typing_extensions import List, TypedDict


class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

In [ ]:
def retrieve(state: State):
    retrieved_docs = vectorstore.similarity_search(state["question"], k=20)
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response.content}

In [ ]:
from langgraph.graph import START, StateGraph

graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
result = graph.invoke({"question": "what is Zelomax?"})

print(f"Context: {result['context']}\n\n")
print(f"Answer: {result['answer']}")

## Alternatives: RAG Workflow without LangGraph

In [ ]:
# --- Alternative: Manual RAG without LangGraph ---
# Minimal workflow: retrieve top-k chunks, build a compact prompt, call the LLM, and show sources.

from langchain.vectorstores import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.embeddings import OpenAIEmbeddings

vectorstore = Chroma.from_documents(documents=chunks, embedding=OpenAIEmbeddings(model="openai.text-embedding-3-large"))

def format_docs(docs):
    return "\n\n---\n\n".join(d.page_content for d in docs)

question = "What is Zelomax?" 
k = 5

# 1) Retrieve
docs = vectorstore.similarity_search(question, k=k)

# 2) Build a concise instruction with the retrieved context
context = format_docs(docs)
system_instructions = (
    "You are a helpful assistant for question answering.\n"
    "Use ONLY the provided context to answer concisely (<=3 sentences).\n"
    "If the answer isn't in the context, say you don't know.\n\n"
    f"Context:\n{context}"
)

# 3) Ask the model
response = llm.invoke([
    SystemMessage(content=system_instructions),
    HumanMessage(content=question),
])

# 4) Display answer + sources
print("Answer:\n", response.content, "\n")
print("Sources:\
,
1
    print(f"[{i}] {src}")
